#### Preprocessing Data: `preprocess_features()`

In [ ]:
def load_dataset(X_path="dataset_X.txt", y_path="dataset_y.txt"):
    """Load complex data matrix X (16000×4) and label matrix y (16000×4)."""
    X = np.loadtxt(X_path, dtype=complex)
    y = np.loadtxt(y_path, dtype=int)
    return X, y
# -----------------------------------------------------------
# Convert state string "Q4 Q3 Q2 Q1" → dataset label [q1 q2 q3 q4]
def state_to_label(state_str):
    """Convert a 4-bit string Q4 Q3 Q2 Q1 into dataset label [q1, q2, q3, q4].
    Examples:
        "0001" → (Q4=0,Q3=0,Q2=0,Q1=1) → [1,0,0,0]
        "0010" → (Q4=0,Q3=0,Q2=1,Q1=0) → [0,1,0,0]  """
    if len(state_str) != 4 or any(c not in "01" for c in state_str):
        raise ValueError("State must be 4 bits, e.g. '0100'.")
    # state_str = Q4 Q3 Q2 Q1
    Q4, Q3, Q2, Q1 = (int(b) for b in state_str)
    # dataset labels are [q1, q2, q3, q4] = [Q1, Q2, Q3, Q4]
    return np.array([Q1, Q2, Q3, Q4], dtype=int)
def preprocess_features(X, method):
    """Preprocess real-valued features using different scaling methods.
    Parameters
    X : ndarray, shape (N, d)
        Input feature matrix (real-valued)
    method : str
        'standard' -> StandardScaler
        'robust'   -> RobustScaler
    Returns
    X_scaled : ndarray
        Scaled features
    scaler : fitted scaler object """
    if method == "standard":
        scaler = StandardScaler()
    elif method == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(f"Unknown preprocessing method: {method}")
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler
def encode_features(X_complex, encoding):
    """Encode complex qubit readout data into real-valued features.
    Parameters
    X_complex : ndarray, shape (N,4)
        Complex-valued qubit readouts
    encoding : str
        'iq'    -> [Re1, Im1, Re2, Im2, Re3, Im3, Re4, Im4]
        'angle' -> [|z1|, arg(z1), ..., |z4|, arg(z4)]
    Returns
    X_real : ndarray, shape (N,8)
        Real-valued feature matrix   """
    if encoding == "iq":
        X_real = np.hstack([X_complex.real, X_complex.imag])
    elif encoding == "angle":
        amplitude = np.abs(X_complex)
        phase = np.angle(X_complex)   # range [-pi, pi]
        X_real = np.hstack([amplitude, phase])
    else:
        raise ValueError(f"Unknown encoding method: {encoding}")
    return X_real